In [ ]:
# Import libraries
from transformers import AutoModelForSequenceClassification

In [14]:
# Load the pretrained XLM-RoBERTa sentiment model
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

In [15]:
# Full model architecture
print("Full model architecture:")
print(model)

Full model architecture:
XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_fe

In [16]:
# Configuration details
print("\nConfiguration (pretty):")
print(model.config)
print("\nConfiguration (dict):")
print(model.config.to_dict())


Configuration (pretty):
XLMRobertaConfig {
  "_attn_implementation_autoset": true,
  "architectures": [
    "XLMRobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.51.2",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}


Configuration (dict):
{'return_dict': Tru

In [20]:
# State-dict keys & shapes
sd = model.state_dict()
print(f"\nTotal tensors in state_dict: {len(sd)}")
for k, v in sd.items():
    print(f"{k:<60} -> {tuple(v.shape)}")


Total tensors in state_dict: 201
roberta.embeddings.word_embeddings.weight                    -> (250002, 768)
roberta.embeddings.position_embeddings.weight                -> (514, 768)
roberta.embeddings.token_type_embeddings.weight              -> (1, 768)
roberta.embeddings.LayerNorm.weight                          -> (768,)
roberta.embeddings.LayerNorm.bias                            -> (768,)
roberta.encoder.layer.0.attention.self.query.weight          -> (768, 768)
roberta.encoder.layer.0.attention.self.query.bias            -> (768,)
roberta.encoder.layer.0.attention.self.key.weight            -> (768, 768)
roberta.encoder.layer.0.attention.self.key.bias              -> (768,)
roberta.encoder.layer.0.attention.self.value.weight          -> (768, 768)
roberta.encoder.layer.0.attention.self.value.bias            -> (768,)
roberta.encoder.layer.0.attention.output.dense.weight        -> (768, 768)
roberta.encoder.layer.0.attention.output.dense.bias          -> (768,)
roberta.encode

In [21]:
# Embedding layer info
emb = model.roberta.embeddings.word_embeddings
print(f"\nEmbedding vocab size: {emb.num_embeddings}")
print(f"Embedding dimension:  {emb.embedding_dim}")


Embedding vocab size: 250002
Embedding dimension:  768


In [22]:
# Parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen      = total_params - trainable
print(f"\nTotal parameters:   {total_params:,}")
print(f"Trainable params:   {trainable:,}")
print(f"Frozen params:      {frozen:,}")


Total parameters:   278,045,955
Trainable params:   278,045,955
Frozen params:      0


In [23]:
# Frozen vs. trainable parameters
print("\nFrozen vs. trainable parameters:")
for name, p in model.named_parameters():
    print(f"{name:<60} requires_grad={p.requires_grad}")


Frozen vs. trainable parameters:
roberta.embeddings.word_embeddings.weight                    requires_grad=True
roberta.embeddings.position_embeddings.weight                requires_grad=True
roberta.embeddings.token_type_embeddings.weight              requires_grad=True
roberta.embeddings.LayerNorm.weight                          requires_grad=True
roberta.embeddings.LayerNorm.bias                            requires_grad=True
roberta.encoder.layer.0.attention.self.query.weight          requires_grad=True
roberta.encoder.layer.0.attention.self.query.bias            requires_grad=True
roberta.encoder.layer.0.attention.self.key.weight            requires_grad=True
roberta.encoder.layer.0.attention.self.key.bias              requires_grad=True
roberta.encoder.layer.0.attention.self.value.weight          requires_grad=True
roberta.encoder.layer.0.attention.self.value.bias            requires_grad=True
roberta.encoder.layer.0.attention.output.dense.weight        requires_grad=True
robert

In [25]:
# Inspect the classifier module
print("\nClassifier structure:")
print(model.classifier)


Classifier structure:
XLMRobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=3, bias=True)
)


In [26]:
# Print out label info from the config
print("Number of labels:", model.config.num_labels)
print("ID → label mapping:", model.config.id2label)
print()

Number of labels: 3
ID → label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}



In [27]:
# (Optional) See each parameter name and shape in the head
print("\nHead parameters:")
for name, param in model.classifier.named_parameters():
    print(f"  {name}: {tuple(param.shape)}")


Head parameters:
  dense.weight: (768, 768)
  dense.bias: (768,)
  out_proj.weight: (3, 768)
  out_proj.bias: (3,)
